In [1]:
import pandas as pd
import numpy as np

# ============================================================
# 1) CATÁLOGO — 40 TÍTULOS
# ============================================================
series = pd.DataFrame({
    'item_id': list(range(1, 41)),
    'titulo': [
        # --- Drama / Crime (1-10) ---
        'Breaking Bad', 'The Wire', 'Peaky Blinders', 'The Sopranos', 'Mad Men',
        'True Detective', 'Narcos', 'Ozark', 'Mindhunter', 'Better Call Saul',
        # --- Ficção / Fantasia / Suspense (11-20) ---
        'Stranger Things', 'Game of Thrones', 'Dark', 'Black Mirror', 'Westworld',
        'The Expanse', 'Lost', 'Fringe', 'The Mandalorian', 'Severance',
        # --- Animação (21-26) ---
        'Rick and Morty', 'Bob Esponja', 'Avatar: A Lenda de Aang', 'Futurama',
        'BoJack Horseman', 'Arcane',
        # --- Romance / Drama (27-31) ---
        'Outlander', 'Bridgerton', 'Orgulho e Preconceito', 'Normal People', 'The Crown',
        # --- Comédia / Policial (32-36) ---
        'The Office', 'Brooklyn Nine-Nine', 'Friends', 'Sherlock', 'Parks and Recreation',
        # --- Suspense / Thriller psicológico (37-40) ---
        'Prison Break', 'Dexter', 'You', 'Money Heist'
    ],
    'genero': [
        'Drama','Drama','Drama','Drama','Drama','Drama','Drama','Drama','Drama','Drama',
        'Ficção','Ficção','Ficção','Ficção','Ficção','Ficção','Ficção','Ficção','Ficção','Ficção',
        'Animação','Animação','Animação','Animação','Animação','Animação',
        'Romance','Romance','Romance','Romance','Romance',
        'Comédia','Comédia','Comédia','Comédia','Comédia',
        'Suspense','Suspense','Suspense','Suspense'
    ],
    'subgenero': [
        'Crime','Crime','Crime','Crime','Drama','Suspense','Crime','Suspense','Crime','Crime',
        'Suspense','Fantasia','Mistério','Sci-Fi','Sci-Fi','Sci-Fi','Mistério','Sci-Fi','Aventura','Sci-Fi',
        'Comédia','Comédia','Aventura','Ficção','Drama','Ação',
        'Drama','Histórico','Drama','Drama','Histórico',
        'Mockumentary','Policial','Sitcom','Policial','Mockumentary',
        'Ação','Crime','Thriller','Ação'
    ],
    'ano': [
        2008,2002,2013,1999,2007,2014,2015,2017,2017,2015,
        2016,2011,2017,2011,2016,2015,2004,2008,2019,2022,
        2013,1999,2005,1999,2014,2021,
        2014,2020,1995,2020,2016,
        2005,2013,1994,2010,2009,
        2005,2006,2018,2017
    ],
    'duracao': [
        47,59,60,55,47,55,50,60,55,50,
        51,57,55,50,60,55,45,50,40,50,
        22,11,23,22,25,40,
        55,60,55,50,55,
        22,22,22,88,22,
        45,50,45,50
    ],
    'classificacao': [
        16,16,18,18,16,18,16,16,16,16,
        14,18,16,16,16,16,12,14,14,16,
        16,0,10,12,16,16,
        16,16,12,16,16,
        12,14,12,16,12,
        16,18,16,16
    ],
    'nota_media': [
        4.9,5.0,4.7,4.9,4.7,4.6,4.5,4.4,4.5,4.7,
        4.8,4.7,4.6,4.5,4.4,4.5,4.3,4.2,4.4,4.6,
        4.8,4.6,4.9,4.7,4.7,4.9,
        4.4,4.5,4.6,4.3,4.7,
        4.8,4.7,4.6,4.8,4.5,
        4.3,4.6,4.2,4.4
    ]
})

# ============================================================
# 2) PERFIS DE GOSTO (clusters) — define afinidade por item
# ============================================================
# Cada perfil lista os itens que o usuário tende a consumir.
# A nota base é 4-5; o gerador adiciona ruído e algumas notas 3.

perfis = {
    'crime':      [1,2,3,4,6,7,8,9,10,37,38,39,40],
    'scifi':      [11,12,13,14,15,16,17,18,19,20],
    'animacao':   [21,22,23,24,25,26],
    'romance':    [27,28,29,30,31],
    'comedia':    [32,33,34,35,36,21,22],
    'suspense':   [6,8,9,14,17,37,38,39,40],
    'drama':      [1,4,5,10,27,30,31],
    'fantasia':   [12,16,19,23,26],
}

# 100 usuários distribuídos entre os perfis (mistura realista)
rng = np.random.default_rng(seed=42)
perfil_nomes = list(perfis.keys())
pesos = [0.18, 0.15, 0.12, 0.10, 0.12, 0.12, 0.11, 0.10]  # soma 1.0

usuarios = [f'U{i:03d}' for i in range(1, 101)]
perfil_do_usuario = rng.choice(perfil_nomes, size=100, p=pesos)

# ============================================================
# 3) GERAR INTERAÇÕES (~600 registros)
# ============================================================
registros = []
for user, perfil in zip(usuarios, perfil_do_usuario):
    itens_perfil = perfis[perfil]

    # cada usuário consome entre 4 e 9 itens do seu perfil principal
    n_principal = rng.integers(4, 10)
    itens_escolhidos = rng.choice(itens_perfil, size=min(n_principal, len(itens_perfil)), replace=False)

    # 30% de chance de "explorar" 1-3 itens fora do perfil (descoberta)
    extras = []
    if rng.random() < 0.30:
        n_extra = rng.integers(1, 4)
        candidatos = [i for i in range(1, 41) if i not in itens_escolhidos]
        extras = rng.choice(candidatos, size=n_extra, replace=False)

    for item in list(itens_escolhidos) + list(extras):
        # nota base alta para itens do perfil, mais baixa para exploratórios
        if item in itens_perfil:
            nota = int(rng.choice([5, 5, 4, 4, 3], p=[0.45, 0.25, 0.20, 0.07, 0.03]))
        else:
            nota = int(rng.choice([4, 3, 2], p=[0.30, 0.50, 0.20]))
        registros.append((user, int(item), nota))

interacoes = pd.DataFrame(registros, columns=['user_id', 'item_id', 'nota'])

# Remove duplicatas (mesmo usuário-item) mantendo a última nota
interacoes = interacoes.drop_duplicates(subset=['user_id', 'item_id'], keep='last').reset_index(drop=True)






In [2]:
# ============================================================
# 4) VERIFICAÇÕES
# ============================================================
print("Catálogo:", series.shape)
print(series.head())

print("\nInterações:", interacoes.shape)
print(interacoes.head())

print("\nUsuários únicos:", interacoes['user_id'].nunique())
print("Itens com interação:", interacoes['item_id'].nunique())
print("\nDistribuição de notas:")
print(interacoes['nota'].value_counts().sort_index())

Catálogo: (40, 8)
   item_id          titulo genero subgenero   ano  duracao  classificacao  \
0        1    Breaking Bad  Drama     Crime  2008       47             16   
1        2        The Wire  Drama     Crime  2002       59             16   
2        3  Peaky Blinders  Drama     Crime  2013       60             18   
3        4    The Sopranos  Drama     Crime  1999       55             18   
4        5         Mad Men  Drama     Drama  2007       47             16   

   nota_media  
0         4.9  
1         5.0  
2         4.7  
3         4.9  
4         4.7  

Interações: (664, 3)
  user_id  item_id  nota
0    U001       17     5
1    U001       39     5
2    U001       14     4
3    U001       38     5
4    U001        9     5

Usuários únicos: 100
Itens com interação: 40

Distribuição de notas:
nota
2     13
3     47
4    164
5    440
Name: count, dtype: int64


In [3]:
matriz = interacoes.pivot_table(
    index='user_id', columns='item_id', values='nota'
).fillna(0)

densidade = 100 * (matriz > 0).sum().sum() / matriz.size
print(f"\nMatriz usuário×item: {matriz.shape}")
print(f"Densidade: {densidade:.1f}%  (esparsidade: {100 - densidade:.1f}%)")


Matriz usuário×item: (100, 40)
Densidade: 16.6%  (esparsidade: 83.4%)


# Atividade — Sistema de recomendação com KNN

**Objetivo:** recomendar séries para os usuários `U003`, `U027`, `U057` e `U081` usando K-Nearest Neighbors.

Nesta solução, a abordagem é **baseada em itens**: o algoritmo identifica séries com padrões de avaliação semelhantes. Para cada usuário, as séries ainda não avaliadas recebem uma pontuação calculada a partir da similaridade com as séries que ele já avaliou.

Como os dados são sintéticos, os resultados demonstram o funcionamento do método e não representam preferências reais de espectadores.

In [4]:
from sklearn.neighbors import NearestNeighbors

# A matriz original está organizada como usuários × séries.
# Para encontrar séries parecidas, transpomos a matriz:
# cada linha passa a representar uma série e cada coluna um usuário.
matriz_itens = matriz.T

# Distância cosseno: valores menores indicam perfis de avaliação mais próximos.
knn_itens = NearestNeighbors(metric="cosine", algorithm="brute")
knn_itens.fit(matriz_itens)


def recomendar_series(usuario, vizinhos=10, limite=5):
    """Recomenda séries não avaliadas a partir das séries já avaliadas pelo usuário."""
    notas_usuario = matriz.loc[usuario]
    itens_vistos = notas_usuario[notas_usuario > 0]

    if itens_vistos.empty:
        return pd.DataFrame(columns=[
            "titulo", "genero", "nota_prevista", "series_similares"
        ])

    # Busca séries vizinhas para cada série que o usuário avaliou.
    n_busca = min(vizinhos + 1, len(matriz_itens))
    distancias, indices = knn_itens.kneighbors(
        matriz_itens.loc[itens_vistos.index],
        n_neighbors=n_busca
    )

    # Agrega evidências de similaridade para cada série candidata.
    soma_ponderada = {}
    soma_similaridades = {}
    apoio = {}

    for serie_origem, nota_origem, dist_row, idx_row in zip(
        itens_vistos.index, itens_vistos.values, distancias, indices
    ):
        for distancia, idx_item in zip(dist_row, idx_row):
            serie_candidata = matriz_itens.index[idx_item]

            # Ignora a própria série e qualquer item já avaliado.
            if serie_candidata == serie_origem or notas_usuario[serie_candidata] > 0:
                continue

            similaridade = max(0.0, 1.0 - float(distancia))
            if similaridade <= 0:
                continue

            # A nota atribuída à série de origem contribui para a previsão,
            # ponderada pela semelhança entre as duas séries.
            soma_ponderada[serie_candidata] = (
                soma_ponderada.get(serie_candidata, 0.0)
                + similaridade * float(nota_origem)
            )
            soma_similaridades[serie_candidata] = (
                soma_similaridades.get(serie_candidata, 0.0) + similaridade
            )
            apoio[serie_candidata] = apoio.get(serie_candidata, 0) + 1

    if not soma_ponderada:
        return pd.DataFrame(columns=[
            "titulo", "genero", "nota_prevista", "series_similares"
        ])

    previsoes = pd.DataFrame({
        "item_id": list(soma_ponderada.keys()),
        "nota_prevista": [
            soma_ponderada[item] / soma_similaridades[item]
            for item in soma_ponderada
        ],
        "series_similares": [apoio[item] for item in soma_ponderada]
    })

    # Recupera informações do catálogo e organiza o ranking.
    resultado = previsoes.merge(series, on="item_id", how="left")
    resultado = resultado.sort_values(
        ["nota_prevista", "series_similares", "nota_media"],
        ascending=[False, False, False]
    ).head(limite)

    resultado["nota_prevista"] = resultado["nota_prevista"].round(2)
    return resultado[
        ["titulo", "genero", "nota_prevista", "series_similares"]
    ]


usuarios_alvo = ["U003", "U027", "U057", "U081"]

for usuario in usuarios_alvo:
    print(f"\nSugestões para {usuario}")
    recomendacoes = recomendar_series(usuario)
    if recomendacoes.empty:
        print("Não há dados suficientes para gerar recomendações.")
    else:
        print(recomendacoes.to_string(index=False))



Sugestões para U003
        titulo   genero  nota_prevista  series_similares
        Narcos    Drama            5.0                 2
      The Wire    Drama            5.0                 1
Peaky Blinders    Drama            5.0                 1
        Dexter Suspense            5.0                 1
   Money Heist Suspense            5.0                 1

Sugestões para U027
          titulo genero  nota_prevista  series_similares
       Westworld Ficção           5.00                 2
Better Call Saul  Drama           4.69                 4
    The Sopranos  Drama           4.66                 4
         Mad Men  Drama           4.66                 4
    Breaking Bad  Drama           4.64                 4

Sugestões para U057
          titulo   genero  nota_prevista  series_similares
    Breaking Bad    Drama           4.66                 4
         Mad Men    Drama           4.65                 4
    The Sopranos    Drama           4.65                 4
Better Call Saul 

## Como interpretar os resultados

- **Nota prevista:** estimativa baseada nas notas do usuário para séries semelhantes; não é uma avaliação real.
- **Séries similares:** quantidade de relações de vizinhança que contribuíram para a recomendação. Um número maior indica mais evidências dentro deste método, mas não garante maior qualidade.
- **Limitações:** a base é simulada e as notas ausentes são representadas por zero na matriz. Em aplicações reais, é importante validar o modelo com dados separados e comparar diferentes configurações de vizinhos.

A abordagem baseada em itens foi escolhida para comparar séries entre si, em vez de procurar usuários com gostos parecidos. Assim, o sistema pode sugerir títulos semelhantes aos que cada pessoa já avaliou positivamente.